# Testing Accessibility Knowledge Across Pythia Model Sizes

## Setup

In [2]:
!pip install transformer_lens

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.3 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=06d455b9c6e6d0dfb6422499dae135857c27dc

In [1]:
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

## Prompts

The following prompts were held constant across all model sizes. They are referenced by variable name in each experiment section.


### Experiment 1: Declarative Prompts

In [2]:
declarative_prompts = [
    "A screen reader is",
    "WCAG stands for",
    "A skip link is",
    "The purpose of alt text is",
    "ARIA stands for",
    "A focus indicator is",
    "Keyboard navigation allows",
    "Color contrast is important because",
    "Semantic HTML helps",
    "Captions are used for",
]

### Experiment 2: Evaluative Prompts

In [3]:
evaluative_prompts = [
    "The following code is not accessible because it doesn't have what? <img src='photo.jpg'>",
    "A <div> with onclick is not accessible because",
    "The accessibility problem with <a href='#'></a> is",
    "<input type='text'> needs a",
    "A button that only says 'Click here' is bad because",
]

## Pythia 160m

### Load Model

In [6]:
model_name = "pythia-160m"

In [7]:
model = HookedTransformer.from_pretrained(model_name)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-160m into HookedTransformer
Layers: 12
Heads: 12
Hidden size: 768
Params: 162.3M


### Model Verification

In [ ]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a device that allows a user to view a screen

Cached 219 different activation points!


### Experiment 1: Declarative Prompts

In [8]:
for prompt in declarative_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a device that allows a user to view a screen
WCAG stands for                →  the International Commission on Geographic Names (ICCG
A skip link is                 →  a link to a page that is not a link
The purpose of alt text is     →  to provide a means for the user to communicate with
ARIA stands for                →  the first time in the history of the country,
A focus indicator is           →  a set of indicators that can be used to measure
Keyboard navigation allows     →  you to navigate through the world with ease.

Color contrast is important because →  it is the most important factor in determining the color
Semantic HTML helps            →  you understand the content of your website.


Captions are used for          →  the following purposes:

The following is a


### Experiment 2: Evaluative Prompts

In [9]:
for prompt in evaluative_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

I'm trying to create a simple image that will be displayed on a page. I'm
A <div> with onclick is not accessible because →  it is not a <div>
<div> with onclick is not accessible because it is not
The accessibility problem with <a href='#'></a> is →  that it is not possible to access the content of the page.

The accessibility problem with <
<input type='text'> needs a    →  password
<input type='password' name='password' value='password' />
<input
A button that only says 'Click here' is bad because →  it's not a button.

A button that only says 'Click here' is bad because


### Experiment 3: Perplexity

In [ ]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()

    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 106.73429107666016
Wrong: 41.367984771728516


### Experiment 4: Attention Pattern Analysis

In [10]:
import csv
import torch

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("160m_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 160m_attention_binding.csv")

Found 46 heads above threshold 0.1

Top 10 by binding strength:
Layer 11, Head  8: 1.0
Layer 11, Head  6: 0.9712
Layer  3, Head  0: 0.9243
Layer  3, Head  2: 0.9156
Layer  1, Head  1: 0.7337
Layer  7, Head  7: 0.6743
Layer  0, Head  0: 0.6281
Layer  1, Head  2: 0.6271
Layer  2, Head 11: 0.6195
Layer  2, Head  8: 0.5459
Saved to 160m_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 1 Head 8**

In [4]:
!pip install circuitsvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.9 MB/s eta 0:00:00


In [11]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 11
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [ ]:
# Run this between models
import torch
import gc
del model_name
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 410m

### Load Model

In [14]:
model_name = "pythia-410m"

In [15]:
model = HookedTransformer.from_pretrained(model_name)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/911M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-410m into HookedTransformer
Layers: 24
Heads: 16
Hidden size: 1024
Params: 405.3M


### Model Verification

In [16]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a device that allows a user to view a screen

Cached 435 different activation points!


### Experiment 1: Declarative Prompts

In [18]:
for prompt in declarative_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a device that allows a user to view a screen
WCAG stands for                →  the World Confederation of Agricultural and Food Industries.
A skip link is                 →  a link that is not part of the main page
The purpose of alt text is     →  to provide a way for users to customize the text
ARIA stands for                →  the acronym for the acronym for
A focus indicator is           →  a device that is used to indicate the presence of
Keyboard navigation allows     →  you to navigate through the various sections of the site
Color contrast is important because →  it can be used to distinguish between different colors.
Semantic HTML helps            →  you create a rich, interactive website.


Captions are used for          →  the purpose of providing a clear and concise description of


### Experiment 2: Evaluative Prompts

In [19]:
for prompt in evaluative_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not a valid HTML element.

A <div> with onclick is not accessible because
The accessibility problem with <a href='#'></a> is →  that it is not clear what the user is looking for.

The <a href='#'
<input type='text'> needs a    →  <input type='text'>

<input type='submit'>

</form>

A button that only says 'Click here' is bad because →  it's not a link.

A button that only says 'Click here' is bad because


### Experiment 3: Perplexity

In [20]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()

    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 40.05071258544922
Wrong: 32.8038444519043


### Experiment 4: Attention Pattern Analysis

In [21]:
import csv
import torch

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("410m_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 410m_attention_binding.csv")

Found 71 heads above threshold 0.1

Top 10 by binding strength:
Layer  0, Head 14: 0.9758
Layer  3, Head 13: 0.9591
Layer  1, Head  3: 0.9404
Layer  3, Head  3: 0.8673
Layer  5, Head  2: 0.8562
Layer  3, Head  4: 0.8519
Layer  0, Head  0: 0.8454
Layer  1, Head 10: 0.8005
Layer  3, Head  5: 0.7801
Layer  2, Head 13: 0.7638
Saved to 410m_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 0 Head 14**

In [22]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 0
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [23]:
# Run this between models
import torch
import gc
del model_name
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 1b

### Load Model

In [24]:
model_name = "pythia-1b"

In [25]:
model = HookedTransformer.from_pretrained(model_name)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.09G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-1b into HookedTransformer
Layers: 16
Heads: 8
Hidden size: 2048
Params: 1011.7M


### Model Verification

In [26]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a program that reads text from a computer screen and

Cached 291 different activation points!


### Experiment 1: Declarative Prompts

In [27]:
for prompt in declarative_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a program that reads text from a computer screen and
WCAG stands for                →  what?

The World Council for Children’
A skip link is                 →  a type of data link used in a computer network
The purpose of alt text is     →  to provide a way to add text to an image
ARIA stands for                →  “Artificial Replacement of a Human Being”.
A focus indicator is           →  a device that is used to indicate the focus of
Keyboard navigation allows     →  you to navigate through the pages of this website.
Color contrast is important because →  it is the basis for many of the visual effects
Semantic HTML helps            →  you to create a website that is easy to read
Captions are used for          →  a variety of purposes, including providing information to a


### Experiment 2: Evaluative Prompts

In [28]:
for prompt in evaluative_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not a <span> element.

A <div> with onclick is not accessible
The accessibility problem with <a href='#'></a> is →  that it is not a valid HTML tag.

The <a href='#'></a>
<input type='text'> needs a    →  value
<input type='text'> needs a value
<input type='text'> needs a
A button that only says 'Click here' is bad because →  it's a clickable link.

A button that only says 'Click here' is bad


### Experiment 3: Perplexity

In [ ]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()

    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 18.804914474487305
Wrong: 42.17985153198242


### Experiment 4: Attention Pattern Analysis

In [29]:
import csv
import torch

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("1b_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 1b_attention_binding.csv")

Found 28 heads above threshold 0.1

Top 10 by binding strength:
Layer  3, Head  5: 0.9932
Layer  1, Head  4: 0.9704
Layer  0, Head  3: 0.9656
Layer  1, Head  0: 0.8892
Layer  3, Head  6: 0.7256
Layer  2, Head  2: 0.7
Layer  1, Head  3: 0.6673
Layer  2, Head  0: 0.661
Layer  6, Head  3: 0.645
Layer  1, Head  2: 0.5844
Saved to 1b_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 3 Head 5**

In [31]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 3
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [32]:
# Run this between models
import torch
import gc
del model_name
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 2.8b

### Load Model

In [33]:
model_name = "pythia-2.8b"

In [34]:
model = HookedTransformer.from_pretrained(model_name)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-2.8b into HookedTransformer
Layers: 32
Heads: 32
Hidden size: 2560
Params: 2774.9M


### Model Verification

In [35]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a software application that reads aloud the text on a

Cached 579 different activation points!


### Experiment 1: Declarative Prompts

In [36]:
for prompt in declarative_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a software application that reads aloud the text on a
WCAG stands for                →  the World Council of Churches. The WCC
A skip link is                 →  a link that is used to skip a section of
The purpose of alt text is     →  to provide a brief description of the image.

ARIA stands for                →  “A Rational Approach to Information and Automation
A focus indicator is           →  a device that is used to indicate the focus of
Keyboard navigation allows     →  you to navigate through the pages of this website.
Color contrast is important because →  it is a key factor in determining the quality of
Semantic HTML helps            →  you create semantic HTML documents.

Semantic
Captions are used for          →  the purpose of identifying the subject of a photograph.


### Experiment 2: Evaluative Prompts

In [37]:
for prompt in evaluative_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not in the same document as the <a> that is clicked.

A <
The accessibility problem with <a href='#'></a> is →  that it is not a valid HTML tag.

The accessibility problem with <a href='#'
<input type='text'> needs a    →  value

<input type='text' value=''>

<input type='text'
A button that only says 'Click here' is bad because →  it's not clear what the button does.

A button that says 'Click here' is


### Experiment 3: Perplexity

In [38]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()

    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 13.648371696472168
Wrong: 54.90732192993164


### Experiment 4: Attention Pattern Analysis

#### Screen Reader

In [39]:
import csv
import torch

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("2.8b_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 2.8b_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]
Found 101 heads above threshold 0.1

Top 10 by binding strength:
Layer  1, Head 12: 0.9909
Layer  1, Head 29: 0.984
Layer  1, Head  6: 0.9815
Layer  0, Head 21: 0.9726
Layer  1, Head 17: 0.9719
Layer  1, Head 25: 0.9483
Layer  1, Head 11: 0.9268
Layer  1, Head 22: 0.9058
Layer 29, Head  7: 0.9019
Layer  4, Head 16: 0.8896
Saved to 2.8b_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 1 Head 12**

In [40]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 1
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Alt Text

In [41]:
import csv
import torch

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 5
        screen_idx = 4
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("2.8b_alttext_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 2.8b_alttext_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'An'), (2, ' image'), (3, ' needs'), (4, ' alt'), (5, ' text'), (6, ' to'), (7, ' be'), (8, ' accessible')]
Found 200 heads above threshold 0.1

Top 10 by binding strength:
Layer  1, Head 25: 0.9899
Layer  3, Head  1: 0.9875
Layer  1, Head 12: 0.9844
Layer  1, Head 16: 0.9598
Layer  3, Head 24: 0.945
Layer  3, Head 20: 0.9275
Layer 21, Head  6: 0.9165
Layer  1, Head  6: 0.8808
Layer  4, Head 16: 0.8742
Layer 25, Head  2: 0.8689
Saved to 2.8b_alttext_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 1 Head 25**

In [42]:
import circuitsvis as cv

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 1
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Skip Link

In [43]:
import csv
import torch

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 4
        screen_idx = 3
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("2.8b_skiplink_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 2.8b_skiplink_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'Use'), (2, ' a'), (3, ' skip'), (4, ' link'), (5, ' to'), (6, ' bypass'), (7, ' navigation')]
Found 182 heads above threshold 0.1

Top 10 by binding strength:
Layer 30, Head  4: 0.9949
Layer  1, Head 12: 0.9839
Layer 30, Head 29: 0.9671
Layer 28, Head  1: 0.9452
Layer  1, Head  6: 0.9212
Layer  4, Head 16: 0.8858
Layer  3, Head 23: 0.8716
Layer 27, Head  9: 0.8458
Layer  6, Head  6: 0.8367
Layer 27, Head 28: 0.8273
Saved to 2.8b_skiplink_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 30 Head 4**

In [45]:
import circuitsvis as cv

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 30
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [46]:
# Run this between models
import torch
import gc
del model_name
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 6.9b

### Load Model

In [47]:
model_name = "pythia-6.9b"

In [48]:
model = HookedTransformer.from_pretrained(model_name)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-6.9b into HookedTransformer
Layers: 32
Heads: 32
Hidden size: 4096
Params: 6856.8M


### Model Verification

In [49]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a software application that reads text on a computer screen

Cached 579 different activation points!


### Experiment 1: Declarative Prompts

In [50]:
for prompt in declarative_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a software application that reads text on a computer screen
WCAG stands for                →  Web Content Accessibility Guidelines. These guidelines are a
A skip link is                 →  a link that is not displayed in the browser.
The purpose of alt text is     →  to provide a description of the image.


ARIA stands for                →  the Association of Research Libraries in Africa. It
A focus indicator is           →  a device that is used to indicate the focus of
Keyboard navigation allows     →  you to navigate through the pages of this website.
Color contrast is important because →  it helps us to distinguish objects in our environment.
Semantic HTML helps            →  you to create semantic HTML. It is a tool
Captions are used for          →  a variety of purposes. They can be used to


### Experiment 2: Evaluative Prompts

In [51]:
for prompt in evaluative_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not a form control.

<div onclick="alert('hello')">


The accessibility problem with <a href='#'></a> is →  that it is not a valid HTML element.

The accessibility problem with <a href='#'
<input type='text'> needs a    →  value

A button that only says 'Click here' is bad because →  it's not clear what the button does.

A button that says 'Click here' is


### Experiment 3: Perplexity

In [52]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()

    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 15.625739097595215
Wrong: 46.063819885253906


### Experiment 4: Attention Pattern Analysis

In [53]:
import csv
import torch

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

threshold = 0.1  # adjust if you want more or fewer results
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]  # [heads, seq, seq]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]  # [seq, seq]
        # find "reader" -> "screen" attention
        # tokens are typically [BOS, 'A', ' screen', ' reader', ' is']
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

with open("6.9b_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "reader_to_screen"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")
print("Saved to 6.9b_attention_binding.csv")

Found 123 heads above threshold 0.1

Top 10 by binding strength:
Layer  2, Head 18: 0.986
Layer  2, Head  8: 0.9693
Layer  2, Head 10: 0.9693
Layer  1, Head  6: 0.9601
Layer  3, Head 14: 0.9409
Layer  3, Head 23: 0.9369
Layer  1, Head 20: 0.9126
Layer  1, Head  8: 0.9064
Layer  1, Head 18: 0.8696
Layer  2, Head 16: 0.8577
Saved to 6.9b_attention_binding.csv


*Circuitsvis* is run on the layer that has the head with the top binding strength.  
**Layer 2 Head 18**

In [ ]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 2
attention = cache["pattern", layer]  # shape: [heads, seq, seq]

cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete model and clear cache

In [54]:
# Run this between models
import torch
import gc
del model_name
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared
